In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif']=['SimHei']
rcParams['axes.unicode_minus']=False
df=pd.read_csv('../data/UserBehavior_sample.csv')
#查看一下数据的结构
display(df.head(1))
display(df.info())
print(f"清洗前的数据{len(df)}条")
#修改数据的类型
df['behavior_type']=df['behavior_type'].astype('category') #把它弄成分类变量
"""
天池官方数据说明里写明了时间戳以UTC存储，而用户行为发生在中国时区，因此需要+8小时转换为北京时间，这是该数据集的标准处理流程
"""
#将时间戳的类型变为日期类型
df['timestamp']=pd.to_datetime(df['timestamp'],unit='s')+ pd.Timedelta(hours=8)
#再次查看数据类型，看是否修改成功
print(df.info())
display(df.head(1))
#查看数据预处理前的描述性统计
display(df.describe().T)

#数据预处理的过程
#查看是否有重复值
dup_values=df.duplicated().sum()
if dup_values>0:
    print(f"该数据集出现了{dup_values}条重复值，应当删除!")
    df.drop_duplicates(inplace=True) #这条语句可以立即执行，不用再复制
    print(f"处理之后的数据形状{df.shape}")
else:
    print("没有重复值！\n")
#创建数据备份
df_original=df.copy()
print(f"查看处理缺失值之前的数据形状:{df.shape}")
#查看各列的缺失值的情况
print("查看各列的缺失值的情况\n")
#处理缺失值
missing_summary={}
for col in df.columns:
    missing_count=df[col].isna().sum()
    if missing_count>0:
        missing_summary[col]={
            "缺失数量":missing_count,
            "缺失比例":(missing_count/len(df[col])*100).round(2),
            "数据类型":df[col].dtype
        }
    else:
        print(f"{col}这列没有数据缺失!")
if missing_summary:
    missing_df=pd.DataFrame(missing_summary).T
    for col,info in missing_df.iterrows():
        dtype=info['数据类型']
        missing_pct=info['缺失比例']
        #缺失比例小于5%，删除缺失行
        if missing_pct<5:
            print(f"{col}：缺失比例{missing_pct}%，小于5%，删除缺失行")
            df=df.dropna(subset=col)
        #数值型：用中位数填充
        elif dtype in ["int64","float64"]:
            median_val=df[col].median()
            print(f"{col}:数值类型变量，使用中位数{median_val:.2f}进行填充")
            df[col]=df[col].fillna(median_val)
        #分类型：用众数进行填充
        elif dtype in ["datetime64[s]","category"]:
            mode_val=df[col].mode()
            print(f"{col}:分类型变量,使用众数{mode_val}填充")
            df[col]=df[col].fillna(mode_val)
else:
    print("整个数据集没有缺失值，不用做数据处理!")

#异常值的处理
#在异常值处理之前，说明一下，在天池数据集说明内容中，要求时间戳在2017年11月25日至2017年12月3日之间，因此不用IQR，也不能用IQR
#删除时间范围外的异常值
start_time=pd.Timestamp('2017-11-25 00:00:00')
end_time=pd.Timestamp('2017-12-03 23:59:59')
#查看之前有多少条记录
before=len(df)
df=df[(df['timestamp']>=start_time) & (df['timestamp']<=end_time)]
print(f"剔除异常时间记录{before-len(df)}条,还剩下{len(df)}条")

#行为类型值域校验，官方给出的四种类型，多余的类型要删除。类型包括('pv', 'buy', 'cart', 'fav')
valid_types=['pv', 'buy', 'cart', 'fav']
bad_count=(~df['behavior_type'].isin(valid_types)).sum()
print(f"非法行为类型记录数:{bad_count}")
if bad_count>0:
    df=df[df['behavior_type'].isin(valid_types)]
    print(f"已剔除非法记录,剩余{len(df)}条记录")
else:
    print("没有非法记录，不用剔除!")

#弄一个年月日时间特征衍生，为后面分析做准备
df['date']=df['timestamp'].dt.date
#将清洗后的数据集转存下来
df.to_csv('../data/UserBehavior_clean.csv',index=False)

,user_id,item_id,category_id,behavior_type,timestamp
0,1000028,1850341,2025483,pv,1511684583


<class 'pandas.DataFrame'>
RangeIndex: 1015568 entries, 0 to 1015567
Data columns (total 5 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   user_id        1015568 non-null  int64
 1   item_id        1015568 non-null  int64
 2   category_id    1015568 non-null  int64
 3   behavior_type  1015568 non-null  str  
 4   timestamp      1015568 non-null  int64
dtypes: int64(4), str(1)
memory usage: 38.7 MB


None

清洗前的数据1015568条
<class 'pandas.DataFrame'>
RangeIndex: 1015568 entries, 0 to 1015567
Data columns (total 5 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   user_id        1015568 non-null  int64         
 1   item_id        1015568 non-null  int64         
 2   category_id    1015568 non-null  int64         
 3   behavior_type  1015568 non-null  category      
 4   timestamp      1015568 non-null  datetime64[us]
dtypes: category(1), datetime64[us](1), int64(3)
memory usage: 32.0 MB
None


,user_id,item_id,category_id,behavior_type,timestamp
0,1000028,1850341,2025483,pv,2017-11-26 16:23:03


,count,mean,min,25%,50%,75%,max,std
user_id,1015568.0,510502.966946,35.0,253888.0,513662.0,764356.0,1017782.0,294784.294524
item_id,1015568.0,2582248.86695,4.0,1300298.0,2583780.0,3862802.0,5163067.0,1487845.483198
category_id,1015568.0,2689596.938923,2171.0,1320293.0,2640118.0,4145813.0,5161669.0,1462206.634136
timestamp,1015568,2017-11-29 20:58:17.549964,2016-10-29 18:35:41,2017-11-27 13:22:37,2017-11-29 21:48:49,2017-12-02 09:57:22.250000,2017-12-04 05:20:27,NaN


该数据集出现了2条重复值，应当删除!
处理之后的数据形状(1015566, 5)
查看处理缺失值之前的数据形状:(1015566, 5)
查看各列的缺失值的情况

user_id这列没有数据缺失!
item_id这列没有数据缺失!
category_id这列没有数据缺失!
behavior_type这列没有数据缺失!
timestamp这列没有数据缺失!
整个数据集没有缺失值，不用做数据处理!
剔除异常时间记录511条,还剩下1015055条
非法行为类型记录数:0
没有非法记录，不用剔除!
